In [2]:
!pip install -U transformers accelerate bitsandbytes sentencepiece pandas tqdm

Defaulting to user installation because normal site-packages is not writeable


In [3]:
import os
import re
import random
import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    set_seed
)

In [7]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: NVIDIA A100 80GB PCIe


In [8]:
import transformers
import bitsandbytes

print(transformers.__version__)
print(bitsandbytes.__version__)

5.14.1
0.49.2


In [10]:
from pathlib import Path

BASE_DIR = Path.cwd().parent

TRAIN_PATH = BASE_DIR / "RUHSOLD_train.tsv"
VAL_PATH = BASE_DIR / "RUHSOLD_validation.tsv"
TEST_PATH = BASE_DIR / "RUHSOLD_test.tsv"

print(TRAIN_PATH)

/home/jovyan/project work/data_analyssis/RUHSOLD_train.tsv


In [11]:
import gc
import torch

# Release the current model from GPU and RAM.
del model
del tokenizer

gc.collect()
torch.cuda.empty_cache()

print("GPU cache cleared.")

GPU cache cleared.


In [12]:
!rm -rf /home/jovyan/.cache/huggingface/hub/models--mistralai--Mistral-7B-Instruct-v0.2
!rm -rf /home/jovyan/.cache/huggingface/hub/.locks/models--mistralai--Mistral-7B-Instruct-v0.2

In [13]:
!df -h /home/jovyan

Filesystem      Size  Used Avail Use% Mounted on
/dev/rbd8        25G  8.1G   17G  33% /home/jovyan


In [14]:
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto",
    dtype=torch.bfloat16
)

model.eval()

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Model loaded successfully.")
print("Model:", MODEL_NAME)
print("Device:", model.device)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Model loaded successfully.
Model: Qwen/Qwen2.5-7B-Instruct
Device: cuda:0


In [15]:
train_df = pd.read_csv(
    TRAIN_PATH,
    sep="\t"
)
print(train_df.shape)
print(train_df.head())
print(train_df.columns.tolist())

(6407, 2)
  kia howa hai aap ko allah bless and protect you  aameen  1
0                                         randdi hai       3
1                              smjh to agai thi mjhy       1
2  haan yrr tuny sahi thukayi ki abhi tak lund da...       3
3              rundi ka bacha bharwaaa ptm ka kuttaa       0
4  kbi b nai ay ga vo bcz phr phr yahodi naraz ho...       1
['kia howa hai aap ko allah bless and protect you  aameen', '1']


In [16]:
print(train_df.columns)
print(train_df.head())

Index(['kia howa hai aap ko allah bless and protect you  aameen', '1'], dtype='str')
  kia howa hai aap ko allah bless and protect you  aameen  1
0                                         randdi hai       3
1                              smjh to agai thi mjhy       1
2  haan yrr tuny sahi thukayi ki abhi tak lund da...       3
3              rundi ka bacha bharwaaa ptm ka kuttaa       0
4  kbi b nai ay ga vo bcz phr phr yahodi naraz ho...       1


In [17]:
train_df = pd.read_csv(
    TRAIN_PATH,
    sep="\t",
    header=None,
    names=["text", "label"]
)

print(train_df.shape)
print(train_df.columns)
print(train_df.head())

(6408, 2)
Index(['text', 'label'], dtype='str')
                                                text  label
0  kia howa hai aap ko allah bless and protect yo...      1
1                                         randdi hai      3
2                              smjh to agai thi mjhy      1
3  haan yrr tuny sahi thukayi ki abhi tak lund da...      3
4              rundi ka bacha bharwaaa ptm ka kuttaa      0


In [18]:
train_df["label"] = pd.to_numeric(
    train_df["label"],
    errors="raise"
).astype(int)

In [19]:
print(train_df["label"].value_counts().sort_index())

label
0    1537
1    3423
2     500
3     537
4     411
Name: count, dtype: int64


In [52]:
LABEL_NAMES = {
    0: "Abusive/Offensive",
    1: "Normal",
    2: "Religious Hate",
    3: "Sexism",
    4: "Profane"
}

In [53]:
TARGET_CLASSES = {
    2: "Religious Hate",
    3: "Sexism",
    4: "Profane"
}

In [54]:
TEXT_COLUMN = "text"
LABEL_COLUMN = "label"

import random

CLASS_WORD_LIMITS = {
    2: (8, 31),   # Religious Hate
    3: (8, 23),   # Sexism
    4: (8, 19)    # Profane
}

def sample_demonstrations(
    dataframe,
    class_id,
    number_of_examples=5,
    random_state=None
):
    min_words, max_words = CLASS_WORD_LIMITS[class_id]

    class_data = dataframe[
        dataframe["label"] == class_id
    ].copy()

    class_data["word_count"] = (
        class_data["text"]
        .astype(str)
        .str.split()
        .str.len()
    )

    eligible_data = class_data[
        class_data["word_count"].between(
            min_words,
            max_words
        )
    ].drop_duplicates(
        subset=["text"]
    )

    if len(eligible_data) < number_of_examples:
        raise ValueError(
            f"Only {len(eligible_data)} eligible examples "
            f"found for class {class_id}."
        )

    demonstrations = eligible_data.sample(
        n=number_of_examples,
        random_state=random_state
    )["text"].tolist()

    return demonstrations

In [55]:
sexism_examples = sample_demonstrations(
    train_df,
    class_id=3,
    number_of_examples=5,
    random_state=42
)

for i, example in enumerate(sexism_examples, start=1):
    print(f"{i}. {example}")

1. khan sib waqae ahmaq hi kyo ki ap jesi makar awrt si shade jo kye the😢😢
2. bagrat ourat hain allah karein aska hal bhi qandeel baloch ki tara ho aur inshallah aska yahi hal hoga 👍👍👍
3. kya haal hain maya jaalon ki rundi....? 🖕🖕🖕🖕🖕🖕
4. begairat sasti kanjri gulbukhari k sath ek sasti randi 😃 😃 😃 😃
5. ooooh bc deepika randi ki ma chodunga thode din ruko main btata hoon inko


In [56]:
CLASS_DESCRIPTIONS = {
    2: {
        "label": "Religious hate speech",
        "definition": (
            "A post that insults, humiliates, stereotypes, or expresses hostility "
            "towards a person or group because of religion, sect, or religious identity."
        )
    },
    3: {
        "label": "Sexist abusive language",
        "definition": (
            "A post that insults, humiliates, stereotypes, or degrades a person "
            "because of gender."
        )
    },
    4: {
        "label": "Profane language",
        "definition": (
            "A post containing vulgar, obscene, or strongly offensive language, "
            "without necessarily targeting religion or gender."
        )
    }
}

In [57]:
def build_generation_prompt(class_id, demonstrations):
    """
    Build a few-shot prompt for generating one synthetic Roman Urdu post.

    Parameters
    ----------
    class_id : int
        RUHSOLD target label: 2, 3, or 4.

    demonstrations : list[str]
        Real training examples belonging to the target class.

    Returns
    -------
    list[dict]
        Chat-formatted message accepted by the Mistral tokenizer.
    """

    if class_id not in CLASS_DESCRIPTIONS:
        raise ValueError(
            f"Unsupported class_id: {class_id}. "
            f"Expected one of {list(CLASS_DESCRIPTIONS.keys())}."
        )

    if not demonstrations:
        raise ValueError("At least one demonstration is required.")

    class_label = CLASS_DESCRIPTIONS[class_id]["label"]
    class_definition = CLASS_DESCRIPTIONS[class_id]["definition"]

    formatted_examples = "\n\n".join(
        f"Example {index}:\n{text.strip()}"
        for index, text in enumerate(demonstrations, start=1)
    )

    user_prompt = f"""
You are an expert data synthesis assistant for hate speech classification datasets.

You are generating synthetic Roman Urdu social media posts for an academic research dataset.

Dataset label:
{class_label}.

Definition:
{class_definition}

Roman Urdu refers to Urdu written using the English Latin alphabet.

Below are real examples from the dataset.

{formatted_examples}

Generate ONE new Roman Urdu social media post that belongs to the same dataset label.

The generated post should preserve the informal writing style, vocabulary, and linguistic characteristics of the examples while expressing a new idea.

The generated post must be different from all provided examples and must not copy or closely paraphrase any example.

Write primarily in Roman Urdu using the English Latin alphabet. Natural English code-mixing is acceptable when it occurs naturally.

Return only the generated post.
""".strip()

    messages = [
        {
            "role": "user",
            "content": user_prompt
        }
    ]

    return messages

In [58]:
import re


def validate_generated_script(text):
    """
    Reject Arabic/Urdu, Devanagari, replacement characters,
    and non-Latin alphabetic characters.
    Emojis and standard punctuation are allowed.
    """

    text = str(text)

    forbidden_scripts = re.compile(
        r"[\u0600-\u06FF"
        r"\u0750-\u077F"
        r"\u08A0-\u08FF"
        r"\u0900-\u097F]"
    )

    if forbidden_scripts.search(text):
        return False

    # Unicode replacement character usually indicates corrupted output.
    if "\ufffd" in text:
        return False

    # Reject alphabetic characters outside ordinary ASCII Latin letters.
    for character in text:
        if character.isalpha() and not (
            "A" <= character <= "Z"
            or "a" <= character <= "z"
        ):
            return False

    return True

In [59]:
from transformers import set_seed
import torch


def generate_one_sample(
    messages,
    seed=42,
    max_new_tokens=80,
    temperature=0.7,
    top_p=0.9,
    typical_p=0.8,
    repetition_penalty=1.2
):
    """
    Generate one response using the currently loaded instruction model.
    """

    set_seed(seed)

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    model_inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt"
    ).to(model.device)

    input_length = model_inputs["input_ids"].shape[1]

    with torch.inference_mode():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            typical_p=typical_p,
            repetition_penalty=repetition_penalty,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated_tokens = generated_ids[0][input_length:]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

In [60]:
import re

def extract_generated_post(raw_output):
    """
    Extracts the candidate tweet while preserving the raw output separately.
    """

    text = str(raw_output).strip()

    # Keep only the content before a new-line translation or explanation.
    text = text.split("\n")[0].strip()

    # Remove an English translation beginning in parentheses.
    text = re.split(r"\s*\(", text, maxsplit=1)[0].strip()

    # Remove common unwanted prefixes.
    prefixes = [
        "Generated post:",
        "Generated tweet:",
        "New post:",
        "New tweet:",
        "Output:"
    ]

    for prefix in prefixes:
        if text.lower().startswith(prefix.lower()):
            text = text[len(prefix):].strip()

    return re.sub(r"\s+", " ", text)

In [61]:
qwen_sexism_results = []

for seed in [42, 43, 44, 45, 46]:

    demonstrations = sample_demonstrations(
        dataframe=train_df,
        class_id=3,
        number_of_examples=5,
        random_state=seed
    )

    messages = build_generation_prompt(
        class_id=3,
        demonstrations=demonstrations
    )

    raw_output = generate_one_sample(
        messages=messages,
        seed=seed,
        max_new_tokens=80,
        temperature=0.7,
        top_p=0.9,
        typical_p=0.8,
        repetition_penalty=1.2
    )

    candidate = extract_generated_post(raw_output)

    result = {
        "seed": seed,
        "demonstrations": demonstrations,
        "raw_output": raw_output,
        "candidate_post": candidate,
        "valid_script": validate_generated_script(candidate),
        "word_count": len(candidate.split())
    }

    qwen_sexism_results.append(result)

    print("=" * 80)
    print("Seed:", seed)
    print("Generated:", candidate)
    print("Valid script:", result["valid_script"])
    print("Word count:", result["word_count"])

Seed: 42
Generated: kayla khali gandi mein mardan na tere sa zyada maloom hai 😡😡😡
Valid script: True
Word count: 12
Seed: 43
Generated: tujhe kuch nahi pta yaar kyun logon se wafa dene ke liye bhool gaya ho jo bete ki tarah milti ho
Valid script: True
Word count: 21
Seed: 44
Generated: ye waise nahi rehti beta, mere pas sari tareekhon hain tumhein jo laga gaya hai
Valid script: True
Word count: 15
Seed: 45
Generated: tu wajh se zindagiyon mein phirso na jaana, apni bete ko pakistanto se khud hi dard do ga
Valid script: True
Word count: 18
Seed: 46
Generated: bholi randi wahan se jaao, abhi tak chal rahi ho waada hai... mere pas photo ni ho tuan mein
Valid script: True
Word count: 19


In [52]:
def clean_generated_text(text):
    text = str(text).strip()

    unwanted_prefixes = [
        "New sentence:",
        "Sentence:",
        "Output:",
        "Roman Urdu:",
        "Generated sentence:"
    ]

    for prefix in unwanted_prefixes:
        if text.lower().startswith(prefix.lower()):
            text = text[len(prefix):].strip()

    text = text.strip('"').strip("'")
    text = re.sub(r"\s+", " ", text)

    return text

In [53]:
cleaned_sample = clean_generated_text(sample)

print("Raw:", sample)
print("Cleaned:", cleaned_sample)

Raw: meri behan bharatiya janani mandir wali insaan banne ke liye na, tu biwi ka gaddha bana rhe hai 🤮🤮
(My sister, you're supposed to become a Bhar
Cleaned: meri behan bharatiya janani mandir wali insaan banne ke liye na, tu biwi ka gaddha bana rhe hai 🤮🤮 (My sister, you're supposed to become a Bhar
